# ۰۱ | از فایل خام تا کاتالوگ
هر بخش یک سؤال دارد. سلول‌ها گزارش‌های قبلی را می‌خوانند؛ دادهٔ خام تغییر نمی‌کند. بعد از هر خروجی برداشتت را در یک سلول Markdown بنویس.

In [1]:
from pathlib import Path
import sys, json, csv, shutil, subprocess
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'digikala/digikala-products.csv').is_file()), None)
assert root is not None, 'Open this notebook inside project3'
import os
os.chdir(root)
def report(name):
    b = Path(name).read_bytes()
    return b.decode('utf-16' if b.startswith((b'\xff\xfe', b'\xfe\xff')) else 'utf-8-sig')
print(root, sys.executable, sys.version.split()[0], sep='\n')

d:\goal\programming\quera\boot_camp\project3
c:\Users\pourya\miniconda3\envs\btq\python.exe
3.11.16


## آیا محیط و ورودی‌ها درست‌اند؟
مسیر پایتون بالا باید مربوط به btq باشد. سرستون‌ها را بدون بارگذاری میلیون‌ها نظر ببین. علامت !! در خروجی Git یعنی فایل نادیده گرفته شده است.

In [2]:
for path in sorted(Path('digikala').glob('*.csv')):
    with path.open(encoding='utf-8-sig', newline='') as f: print(path.name, next(csv.reader(f)))
print(subprocess.check_output(['git','status','--ignored','--short','--','digikala/*.csv','digikala/*.rar'],text=True))
print('Free GiB:', round(shutil.disk_usage(root).free/2**30,1))
if shutil.which('nvidia-smi'): subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'])

digikala-comments.csv ['id', 'title', 'body', 'created_at', 'rate', 'recommendation_status', 'is_buyer', 'product_id', 'advantages', 'disadvantages', 'likes', 'dislikes', 'seller_title', 'seller_code', 'true_to_size_rate']
digikala-products.csv ['id', 'title_fa', 'Rate', 'Rate_cnt', 'Category1', 'Category2', 'Brand', 'Price', 'Seller', 'Is_Fake', 'min_price_last_month', 'sub_category']
!! digikala/digikala-comments.csv
!! digikala/digikala-products.csv
!! digikala/digikala.rar

Free GiB: 8.7


## در روز اول چه فایل‌های ارزیابی و Mock آماده شده‌اند؟
وجود فایل با موفقیت اجرای آن فرق دارد. تعداد سؤال‌ها را با حداقل ۲۰ در سند مقایسه کن؛ تنوع و درست بودن پاسخ مرجع بررسی جدا می‌خواهد.

In [3]:
for d in ('qbc13_day1','eval'):
    for p in sorted(Path(d).rglob('*')):
        if p.is_file() and '__pycache__' not in p.parts: print(p)
q=Path('eval/day1_queries.jsonl')
if q.exists(): print('Query records:',sum(1 for line in q.read_text(encoding='utf-8-sig').splitlines() if line.strip() and isinstance(json.loads(line),dict)))

qbc13_day1\__init__.py
qbc13_day1\build_eval.py
qbc13_day1\check_mock.py
qbc13_day1\mock_api.py
qbc13_day1\probe.py
eval\day1_queries.jsonl
Query records: 20


## چند نظر داریم و چند نظر به محصول وصل می‌شود؟
خط فیزیکی لزوماً یک رکورد CSV نیست. ممیزی اصلی با csv.reader انجام شد. nan در گزارش برچسب‌ها یک مقدار متنی است، نه کلاس no_idea.

In [4]:
s=report('day1_data_audit.txt'); audit=json.loads(s[s.index('{'):]); print(json.dumps(audit, indent=2))

{
  "products": 1283496,
  "unique_nonblank_product_ids": 948352,
  "missing_product_ids": 0,
  "extra_rows_with_duplicate_product_id": 335144,
  "comments": 6156289,
  "comments_matched_to_product": 6156289,
  "comments_without_product_id": 0,
  "comments_with_unknown_product_id": 0,
  "join_coverage_pct": 100.0,
  "blank_bodies": 0,
  "status_counts": {
    "not_recommended": 504903,
    "recommended": 4162839,
    "nan": 894426,
    "no_idea": 594121
  },
  "elapsed_s": 20.3
}


## چرا اتصال به جدول خام خطرناک است؟
اختلاف تعداد ردیف با تعداد شناسه، تعداد ردیف اضافه است. آمار اختلاف قیمت و فروشنده می‌توانند هم‌پوشانی داشته باشند.

In [5]:
dup=json.loads(report('day1_product_duplicates.txt')); print(json.dumps(dup,indent=2)); assert audit['products']-audit['unique_nonblank_product_ids']==dup['extra_rows']

{
  "extra_rows": 335144,
  "repeated_ids": 258718,
  "identical_only_ids": 246933,
  "any_difference_ids": 11785,
  "title_brand_or_category_diff_ids": 765,
  "price_diff_ids": 8254,
  "seller_diff_ids": 4036
}


## آیا کاتالوگ واقعاً یک ردیف برای هر شناسه دارد؟
برای هر فیلد مقادیر غیرخالی را مقایسه کردیم؛ اگر توافق نداشتند، مقدار قطعی انتخاب نکردیم. اکنون خود فایل Parquet را می‌خوانیم، نه گزارش قبلی.

In [6]:
import duckdb
print(duckdb.sql("""select count(*) as catalog_rows, count(distinct id) as unique_ids, sum(source_rows) as raw_rows, count(*) filter(where identity_conflict) as conflicting_ids from 'data/processed/product_catalog.parquet'"""))

┌──────────────┬────────────┬──────────┬─────────────────┐
│ catalog_rows │ unique_ids │ raw_rows │ conflicting_ids │
│    int64     │   int64    │  int128  │      int64      │
├──────────────┼────────────┼──────────┼─────────────────┤
│       948352 │     948352 │  1283496 │             765 │
└──────────────┴────────────┴──────────┴─────────────────┘



## برای مرور پیاده‌سازی
این سلول فقط کد واقعی پروژه را نشان می‌دهد. نتیجه‌گیری خودت: اگر هر محصول دو فروشنده داشته باشد، چرا نباید قبل از شمارش نظرات آن‌ها را مستقیم به هم وصل کنیم؟

In [7]:
from IPython.display import Code, display
for name in ['count_rows.py','audit_day1.py','audit_product_duplicates.py','build_product_catalog.py']:
    print(name); display(Code(filename=name,language='python'))

count_rows.py


for p in [r"D:/goal/programming/quera/boot_camp/project3/digikala/digikala-products.csv",
          r"D:/goal/programming/quera/boot_camp/project3/digikala/digikala-comments.csv"]:
    with open(p, "rb") as f:
        n = sum(1 for _ in f)
    print(f"{p.split('/')[-1]}: {n:,} lines (incl. header)")

audit_day1.py


"""Run from project3: python audit_day1.py (standard library only)."""
import csv
import json
from collections import Counter
from pathlib import Path
from time import perf_counter


def records(path, required):
    with path.open(encoding='utf-8-sig', newline='') as f:
        reader = csv.reader(f, strict=True)
        header = next(reader)
        ix = [header.index(k) for k in required]
        try:
            for row in reader:
                if not row:  # skip physically empty records, not empty fields
                    continue
                if len(row) != len(header):
                    raise ValueError(f'{path.name}: wrong field count near line {reader.line_num}')
                yield tuple(row[i].strip() for i in ix)
        except csv.Error as err:
            raise ValueError(f'{path.name}: CSV error near line {reader.line_num}: {err}') from err


def main():
    base = Path('digikala')
    start = perf_counter()
    ids = set()
    products = missing = 0
    for (pid,) in records(base / 'digikala-products.csv', ['id']):
        products += 1
        if pid:
            ids.add(pid)
        else:
            missing += 1
    print(f'Products scanned: {products:,}. Reading comments...', flush=True)
    comments = matched = no_pid = no_body = 0
    labels = Counter()
    for pid, label, body in records(base / 'digikala-comments.csv', ['product_id', 'recommendation_status', 'body']):
        comments += 1
        matched += pid in ids
        no_pid += not pid
        no_body += not body
        labels[label] += 1
    print(json.dumps(dict(products=products, unique_nonblank_product_ids=len(ids), missing_product_ids=missing,
        extra_rows_with_duplicate_product_id=products-missing-len(ids), comments=comments,
        comments_matched_to_product=matched, comments_without_product_id=no_pid,
        comments_with_unknown_product_id=comments-matched-no_pid,
        join_coverage_pct=round(100*matched/comments, 2) if comments else None,
        blank_bodies=no_body, status_counts=dict(labels), elapsed_s=round(perf_counter()-start,1)), indent=2))


if __name__ == '__main__':
    main()

audit_product_duplicates.py


"""Run from project3. Read products only; no changes to raw data."""
import csv
import json
from hashlib import blake2b
from pathlib import Path


def fingerprint(values):
    return blake2b(json.dumps(values, ensure_ascii=False, separators=(',', ':')).encode(), digest_size=16).digest()


def audit(path):
    first = {}
    repeated, changed, identity, price, seller = (set() for _ in range(5))
    extra = 0
    with path.open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.reader(stream, strict=True)
        header = next(reader)
        key, cost, shop, *attrs = map(header.index, ('id', 'Price', 'Seller', 'title_fa', 'Brand', 'Category1', 'Category2', 'sub_category'))
        for row in reader:
            if not row:
                continue
            if len(row) != len(header):
                raise ValueError(f'Wrong column count near line {reader.line_num}')
            pid = row[key].strip()
            sig = fingerprint(row)
            if pid not in first:
                first[pid] = (sig, fingerprint([row[i] for i in attrs]), row[cost], row[shop])
                continue
            extra += 1
            repeated.add(pid)
            prev, facts, p, s = first[pid]
            if prev != sig:
                changed.add(pid)
                if facts != fingerprint([row[i] for i in attrs]): identity.add(pid)
                if p != row[cost]: price.add(pid)
                if s != row[shop]: seller.add(pid)
    return dict(extra_rows=extra, repeated_ids=len(repeated), identical_only_ids=len(repeated)-len(changed),
                any_difference_ids=len(changed), title_brand_or_category_diff_ids=len(identity),
                price_diff_ids=len(price), seller_diff_ids=len(seller))


if __name__ == '__main__':
    print(json.dumps(audit(Path('digikala/digikala-products.csv')), indent=2))

build_product_catalog.py


"""Run from project3: python build_product_catalog.py"""
from pathlib import Path
from time import perf_counter
import duckdb

source = Path('digikala/digikala-products.csv')
out = Path('data/processed/product_catalog.parquet')
out.parent.mkdir(parents=True, exist_ok=True)
tmp = out.with_suffix('.tmp.parquet')
fields = ['title_fa', 'Brand', 'Category1', 'Category2', 'sub_category', 'Price', 'Seller', 'Rate', 'Rate_cnt', 'Is_Fake', 'min_price_last_month']
identity = fields[:5]
start = perf_counter()
con = duckdb.connect()
con.execute("SET threads=2")
con.execute("SET memory_limit='3GB'")
# Column names below are fixed in code, not user input. Preserve raw string units.
clean = ', '.join(f'nullif(trim("{k}"), \'\') AS "{k}"' for k in fields)
# MIN=MAX means all non-null observed values agree; no arbitrary first row.
values = ', '.join(f'CASE WHEN min("{k}")=max("{k}") THEN min("{k}") END AS "{k}"' for k in fields)
flags = ', '.join(f'coalesce(min("{k}")<>max("{k}"), false) AS "{k}_conflict"' for k in fields)
ambiguous = ' OR '.join(f'"{k}_conflict"' for k in identity)
sql = f'''WITH clean AS (SELECT trim(id) AS id, {clean}
    FROM read_csv(?, header=true, all_varchar=true, delim=',', quote='"', escape='"')),
 grouped AS (SELECT id, count(*) AS source_rows, {values}, {flags} FROM clean GROUP BY id)
 SELECT *, ({ambiguous}) AS identity_conflict FROM grouped'''
con.sql(sql, params=[str(source)]).write_parquet(str(tmp), compression='zstd')
stats = con.execute('''SELECT count(*), count(DISTINCT id), count(*) FILTER (WHERE id IS NULL OR id=''), sum(source_rows), count(*) FILTER (WHERE identity_conflict) FROM read_parquet(?)''', [str(tmp)]).fetchone()
n, unique, empty, raw, ambiguous_count = stats
if not (n == unique == 948352 and empty == 0 and raw == 1283496):
    raise RuntimeError(f'Catalog validation failed: {stats}')
tmp.replace(out)
print(f'catalog_rows={n} | unique_ids={unique} | source_rows={raw} | identity_conflicts={ambiguous_count}')
print(f'{out} | {out.stat().st_size / 2**20:.1f} MiB | {perf_counter()-start:.1f} s')